# Fase 0 — Analisi Esplorativa e Bias di Genere nel Dataset GOSSIS

**Workshop: Bias di Genere nei Dataset Clinici**

---

Questo notebook guida passo-passo l'esplorazione del dataset **GOSSIS** (Global Open Source Severity of Illness Score), 
un dataset reale utilizzato per la predizione della mortalità ospedaliera in terapia intensiva (ICU).

**Obiettivo**: capire *prima* di costruire qualsiasi modello predittivo se i dati contengono squilibri, 
differenze sistematiche o lacune che potrebbero tradursi in **bias algoritmico** — in particolare rispetto al **genere**.

> *Non serve alcuna esperienza di programmazione per seguire questo notebook: ogni cella di codice è preceduta da una spiegazione in italiano.*

---
## Sezione 1 — Setup e Caricamento Dati

In questa sezione:
- Montiamo Google Drive per accedere ai file
- Importiamo le librerie necessarie
- Carichiamo il dataset principale (`full_data.csv`)
- Diamo una prima occhiata alla struttura dei dati

In [ ]:
# Montaggio di Google Drive (necessario su Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === CONFIGURAZIONE: modifica questo percorso se necessario ===
# Deve puntare alla cartella che contiene full_data.csv, train_features.csv, ecc.
path_to_data_folder = '/content/drive/MyDrive/Workshop_1/Workshop_Dataset/'

In [ ]:
# Importazione delle librerie
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import warnings

# Impostazioni grafiche coerenti per tutto il notebook
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

# Palette coerente: blu per maschi, arancione/rosso per femmine
GENDER_PALETTE = {'M': '#4C72B0', 'F': '#DD8452'}
GENDER_PALETTE_01 = {0: '#4C72B0', 1: '#DD8452'}  # per colonne binarie

print('Librerie caricate correttamente.')

In [ ]:
# Caricamento del dataset completo
df = pd.read_csv(path_to_data_folder + 'full_data.csv')

print(f'Dimensioni del dataset: {df.shape[0]} righe x {df.shape[1]} colonne')
print(f'\nNumero totale di pazienti: {df.shape[0]:,}')

In [ ]:
# Tipi di dato per ogni colonna
df.info(verbose=False)

In [ ]:
# Prime 5 righe del dataset
df.head()

---
## Sezione 2 — Variabile Target: Mortalità Ospedaliera

La variabile che vogliamo predire si chiama **`hospital_death`**:
- **0** = il paziente è sopravvissuto
- **1** = il paziente è deceduto in ospedale

Prima di tutto, vediamo quanto è frequente l'evento «morte» nel dataset. 
Questo è fondamentale perché un forte **sbilanciamento delle classi** influenza come il modello apprende.

In [ ]:
# Distribuzione della variabile target
target_counts = df['hospital_death'].value_counts()
target_pct = df['hospital_death'].value_counts(normalize=True) * 100

print('Distribuzione della mortalità ospedaliera:')
print(f'  Sopravvissuti (0): {target_counts[0]:,} ({target_pct[0]:.1f}%)')
print(f'  Deceduti     (1): {target_counts[1]:,} ({target_pct[1]:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Barplot
sns.countplot(x='hospital_death', data=df, palette=['#2ecc71', '#e74c3c'], ax=axes[0])
axes[0].set_title('Distribuzione della Mortalità Ospedaliera', fontsize=13)
axes[0].set_xlabel('hospital_death (0 = Sopravvissuto, 1 = Deceduto)')
axes[0].set_ylabel('Numero di pazienti')

# Torta
axes[1].pie(target_counts, labels=['Sopravvissuti', 'Deceduti'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Proporzione Sopravvissuti vs Deceduti', fontsize=13)

plt.tight_layout()
plt.show()

### Cosa significa questo sbilanciamento?

Solo circa l'**8.6%** dei pazienti muore in ospedale. Questo vuol dire che un modello "pigro" potrebbe 
semplicemente predire *"sopravvive"* per tutti e ottenere comunque un'accuratezza del 91.4%.

Per questo motivo, quando valuteremo i modelli, **non basterà guardare l'accuratezza**: dovremo usare metriche 
come recall, precision e AUC che tengano conto dello sbilanciamento.

---
## Sezione 3 — Distribuzione per Genere

Analizziamo come il **genere** è distribuito nel dataset e se esistono differenze nella mortalità tra maschi e femmine.

Controlleremo anche che la distribuzione sia coerente tra set di **training** e **test** 
(i file già preprocessati che useremo per addestrare i modelli).

In [ ]:
# Distribuzione per genere
gender_counts = df['gender'].value_counts()
gender_pct = df['gender'].value_counts(normalize=True) * 100

print('Distribuzione per genere:')
for g in gender_counts.index:
    print(f'  {g}: {gender_counts[g]:,} ({gender_pct[g]:.1f}%)')

In [ ]:
# Grafico a barre della distribuzione per genere
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x='gender', data=df, palette=GENDER_PALETTE, order=['M', 'F'], ax=ax)
ax.set_title('Distribuzione per Genere', fontsize=13)
ax.set_xlabel('Genere')
ax.set_ylabel('Numero di pazienti')

# Aggiungiamo le percentuali sopra le barre
for p in ax.patches:
    pct = 100 * p.get_height() / len(df)
    ax.annotate(f'{pct:.1f}%', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Tasso di mortalità per genere
mortality_by_gender = df.groupby('gender')['hospital_death'].mean() * 100

print('Tasso di mortalità per genere:')
for g in mortality_by_gender.index:
    print(f'  {g}: {mortality_by_gender[g]:.2f}%')

In [ ]:
# Grafico: mortalità per genere
fig, ax = plt.subplots(figsize=(6, 4))

mortality_by_gender_df = df.groupby('gender')['hospital_death'].mean().reset_index()
mortality_by_gender_df['hospital_death'] *= 100

sns.barplot(x='gender', y='hospital_death', data=mortality_by_gender_df,
            palette=GENDER_PALETTE, order=['M', 'F'], ax=ax)
ax.set_title('Tasso di Mortalità per Genere', fontsize=13)
ax.set_xlabel('Genere')
ax.set_ylabel('Mortalità (%)')
ax.set_ylim(0, 15)

for p in ax.patches:
    ax.annotate(f'{p.get_height():.2f}%', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Verifica della distribuzione di genere nei set train e test preprocessati
train_features = pd.read_csv(path_to_data_folder + 'train_features.csv')
test_features = pd.read_csv(path_to_data_folder + 'test_features.csv')

# Nei file preprocessati il genere è codificato come colonne one-hot: gender_F, gender_M
print('Distribuzione di genere nel set di TRAINING:')
if 'gender_F' in train_features.columns:
    train_f = train_features['gender_F'].sum()
    train_m = train_features['gender_M'].sum()
    print(f'  M: {int(train_m):,} ({100*train_m/len(train_features):.1f}%)')
    print(f'  F: {int(train_f):,} ({100*train_f/len(train_features):.1f}%)')
else:
    print('  Colonne gender_F/gender_M non trovate. Verifica i nomi delle colonne.')

print(f'\nDistribuzione di genere nel set di TEST:')
if 'gender_F' in test_features.columns:
    test_f = test_features['gender_F'].sum()
    test_m = test_features['gender_M'].sum()
    print(f'  M: {int(test_m):,} ({100*test_m/len(test_features):.1f}%)')
    print(f'  F: {int(test_f):,} ({100*test_f/len(test_features):.1f}%)')

print(f'\nDimensioni: Train = {len(train_features):,} righe, Test = {len(test_features):,} righe')

> **Osservazione**: i maschi rappresentano circa il 54% del campione. Le donne hanno un tasso di mortalità 
> leggermente più alto (circa 8.8% vs 8.4%). Anche se la differenza sembra piccola, su decine di migliaia di 
> pazienti potrebbe diventare clinicamente rilevante — e un modello potrebbe amplificarla.

---
## Sezione 4 — Distribuzione per Etnia

Oltre al genere, l'**etnia** è un'altra dimensione critica per il bias. 
Vediamo come è distribuita nel dataset e come interagisce con genere e mortalità.

In [ ]:
# Distribuzione per etnia
ethnicity_counts = df['ethnicity'].value_counts()
ethnicity_pct = df['ethnicity'].value_counts(normalize=True) * 100

print('Distribuzione per etnia:')
for e in ethnicity_counts.index:
    print(f'  {e}: {ethnicity_counts[e]:,} ({ethnicity_pct[e]:.1f}%)')

In [ ]:
# Grafico distribuzione per etnia
fig, ax = plt.subplots(figsize=(10, 5))
order = ethnicity_counts.index.tolist()
sns.countplot(y='ethnicity', data=df, order=order, palette='Set2', ax=ax)
ax.set_title('Distribuzione per Etnia', fontsize=13)
ax.set_xlabel('Numero di pazienti')
ax.set_ylabel('Etnia')

for p in ax.patches:
    width = p.get_width()
    pct = 100 * width / len(df)
    ax.annotate(f'{int(width):,} ({pct:.1f}%)', (width + 200, p.get_y() + p.get_height() / 2.),
                ha='left', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Tasso di mortalità per etnia
mortality_by_ethnicity = df.groupby('ethnicity')['hospital_death'].mean() * 100
mortality_by_ethnicity = mortality_by_ethnicity.sort_values(ascending=False)

print('Tasso di mortalità per etnia:')
for e in mortality_by_ethnicity.index:
    print(f'  {e}: {mortality_by_ethnicity[e]:.2f}%')

In [ ]:
# Grafico: mortalità per etnia
fig, ax = plt.subplots(figsize=(10, 5))
mortality_eth_df = mortality_by_ethnicity.reset_index()
mortality_eth_df.columns = ['ethnicity', 'mortality_pct']

sns.barplot(y='ethnicity', x='mortality_pct', data=mortality_eth_df, palette='Set2', ax=ax)
ax.set_title('Tasso di Mortalità per Etnia', fontsize=13)
ax.set_xlabel('Mortalità (%)')
ax.set_ylabel('Etnia')
ax.set_xlim(0, 15)

for p in ax.patches:
    width = p.get_width()
    ax.annotate(f'{width:.1f}%', (width + 0.2, p.get_y() + p.get_height() / 2.),
                ha='left', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Tabella incrociata: genere x etnia — conteggi e mortalità
cross_counts = pd.crosstab(df['ethnicity'], df['gender'], margins=True)
print('=== Conteggi: Etnia x Genere ===')
print(cross_counts)
print()

cross_mortality = df.groupby(['ethnicity', 'gender'])['hospital_death'].mean() * 100
cross_mortality_table = cross_mortality.unstack().round(2)
print('=== Tasso di Mortalità (%): Etnia x Genere ===')
print(cross_mortality_table)

> **Osservazione**: il dataset è dominato da pazienti **caucasici** (~78%). Gruppi come 
> Native American e Asian rappresentano meno del 2% ciascuno. Un modello addestrato su questi dati 
> avrà molta meno "esperienza" con le minoranze etniche e potrebbe fare previsioni meno accurate per loro.

---
## Sezione 5 — Distribuzione per Età

L'età è uno dei fattori più importanti per la mortalità in terapia intensiva. 
Vediamo come si distribuisce nel dataset e se ci sono differenze tra i generi.

In [ ]:
# Istogramma dell'età sovrapposto per genere
fig, ax = plt.subplots(figsize=(10, 5))

for g, color in GENDER_PALETTE.items():
    subset = df[df['gender'] == g]['age'].dropna()
    ax.hist(subset, bins=40, alpha=0.5, label=f'{g} (n={len(subset):,})', color=color, edgecolor='white')

ax.set_title('Distribuzione dell\'Età per Genere', fontsize=13)
ax.set_xlabel('Età (anni)')
ax.set_ylabel('Numero di pazienti')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot dell'età per genere
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(x='gender', y='age', data=df, palette=GENDER_PALETTE, order=['M', 'F'], ax=ax)
ax.set_title('Boxplot dell\'Età per Genere', fontsize=13)
ax.set_xlabel('Genere')
ax.set_ylabel('Età (anni)')
plt.tight_layout()
plt.show()

# Statistiche descrittive
print('Statistiche dell\'età per genere:')
print(df.groupby('gender')['age'].describe().round(1))

In [ ]:
# Creiamo fasce di età
bins = [0, 45, 65, 80, 120]
labels = ['<45', '45-65', '65-80', '>80']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)

# Heatmap: mortalità per fascia di età e genere
heatmap_data = df.groupby(['age_group', 'gender'])['hospital_death'].mean() * 100
heatmap_data = heatmap_data.unstack()

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            cbar_kws={'label': 'Mortalità (%)'})
ax.set_title('Tasso di Mortalità (%) per Fascia di Età e Genere', fontsize=13)
ax.set_xlabel('Genere')
ax.set_ylabel('Fascia di età')
plt.tight_layout()
plt.show()

In [ ]:
# Conteggi per fascia di età e genere
age_gender_counts = pd.crosstab(df['age_group'], df['gender'], margins=True)
print('Conteggi per fascia di età e genere:')
print(age_gender_counts)

> **Osservazione**: la mortalità cresce fortemente con l'età. In tutte le fasce, le donne mostrano 
> un tasso di mortalità leggermente diverso dai maschi. Queste differenze saranno importanti quando 
> analizzeremo le previsioni del modello.

---
## Sezione 6 — Analisi dei Valori Mancanti per Genere

I **valori mancanti** (missing values) non sono casuali in ambito clinico. Un dato mancante spesso significa 
che un esame non è stato richiesto o effettuato. Se le donne hanno *più* valori mancanti per certi esami, 
questo potrebbe riflettere un **bias implicito** nell'accesso alle cure diagnostiche.

Questa è una delle analisi più importanti di tutto il notebook.

In [ ]:
# Calcoliamo la percentuale di valori mancanti per ogni feature, separatamente per M e F
df_m = df[df['gender'] == 'M']
df_f = df[df['gender'] == 'F']

missing_m = (df_m.isnull().sum() / len(df_m) * 100).round(2)
missing_f = (df_f.isnull().sum() / len(df_f) * 100).round(2)

missing_comparison = pd.DataFrame({
    'Missing_M (%)': missing_m,
    'Missing_F (%)': missing_f
})
missing_comparison['Differenza (F - M)'] = (missing_comparison['Missing_F (%)'] - missing_comparison['Missing_M (%)']).round(2)
missing_comparison['Media_Missing (%)'] = ((missing_comparison['Missing_M (%)'] + missing_comparison['Missing_F (%)']) / 2).round(2)

# Filtriamo solo le colonne numeriche con almeno qualche valore mancante
missing_comparison = missing_comparison[missing_comparison['Media_Missing (%)'] > 0]
missing_comparison = missing_comparison.sort_values('Media_Missing (%)', ascending=False)

print(f'Features con valori mancanti: {len(missing_comparison)}')
print('\nTop 20 features con più valori mancanti:')
missing_comparison.head(20)

In [ ]:
# Grafico: top 20 features con più valori mancanti, divise per genere
top_missing = missing_comparison.head(20).copy()

fig, ax = plt.subplots(figsize=(12, 8))

x = np.arange(len(top_missing))
width = 0.35

bars_m = ax.barh(x + width/2, top_missing['Missing_M (%)'], width, label='M', color=GENDER_PALETTE['M'], alpha=0.8)
bars_f = ax.barh(x - width/2, top_missing['Missing_F (%)'], width, label='F', color=GENDER_PALETTE['F'], alpha=0.8)

ax.set_yticks(x)
ax.set_yticklabels(top_missing.index, fontsize=9)
ax.set_xlabel('Valori Mancanti (%)')
ax.set_title('Top 20 Features con Più Valori Mancanti — Confronto M vs F', fontsize=13)
ax.legend(fontsize=11)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Features con la maggiore differenza di missing tra M e F
print('Features con la maggiore DIFFERENZA di valori mancanti tra F e M:')
print('(valori positivi = più mancanti per le donne)\n')
diff_sorted = missing_comparison.sort_values('Differenza (F - M)', ascending=False)
print(diff_sorted[['Missing_M (%)', 'Missing_F (%)', 'Differenza (F - M)']].head(15))
print('\n--- Features con più mancanti per gli UOMINI ---')
print(diff_sorted[['Missing_M (%)', 'Missing_F (%)', 'Differenza (F - M)']].tail(15))

### Perché i valori mancanti sono un indicatore di bias

Se osserviamo che certe features cliniche (ad esempio emogasanalisi, lattato, creatinina) hanno 
**percentuali di mancanza diverse tra uomini e donne**, questo potrebbe significare che:

1. **Le donne ricevono meno esami diagnostici** in terapia intensiva (bias nell'accesso ai test)
2. **I protocolli clinici** sono stati sviluppati prevalentemente su pazienti maschi
3. **Il modello** riceverà informazioni più complete per un genere rispetto all'altro

Questa **asimmetria informativa** è una forma di **bias implicito** che si trasferisce direttamente 
nel modello predittivo, perché il modello impara da dati *meno completi* per un sottogruppo.

> Nota: nel dataset GOSSIS, le features relative ai **gas ematici** (blood gas) hanno tassi di mancanza 
> enormi (60-70%), il che riduce significativamente l'informazione disponibile per la predizione.

---
## Sezione 7 — Feature Cliniche Chiave per Genere

Esaminiamo un sottoinsieme di features cliniche particolarmente rilevanti per la terapia intensiva 
e vediamo come si distribuiscono tra maschi e femmine.

In [ ]:
# Selezione delle features cliniche chiave
key_features = [
    'age', 'bmi', 'heart_rate_apache', 'resprate_apache',
    'creatinine_apache', 'bun_apache',
    'gcs_motor_apache', 'gcs_verbal_apache',
    'd1_lactate_max', 'd1_hemaglobin_max',
    'd1_platelets_min', 'd1_glucose_max'
]

# Verifichiamo quali esistono nel dataset
available_features = [f for f in key_features if f in df.columns]
print(f'Features disponibili: {len(available_features)} su {len(key_features)}')
print(available_features)

In [ ]:
# Violin plot per le features cliniche chiave, divise per genere
n_features = len(available_features)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, feat in enumerate(available_features):
    sns.violinplot(x='gender', y=feat, data=df, palette=GENDER_PALETTE,
                   order=['M', 'F'], ax=axes[i], inner='quartile')
    axes[i].set_title(feat, fontsize=11)
    axes[i].set_xlabel('')

# Nascondi assi vuoti
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribuzione delle Features Cliniche Chiave per Genere', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Tabella delle medie per genere
means_by_gender = df.groupby('gender')[available_features].mean().T.round(2)
means_by_gender['Differenza (M - F)'] = (means_by_gender['M'] - means_by_gender['F']).round(2)
means_by_gender['Differenza (%)'] = ((means_by_gender['Differenza (M - F)'] / means_by_gender['F']) * 100).round(1)

print('=== Medie delle features cliniche chiave per genere ===')
means_by_gender

> **Osservazione**: alcune features mostrano differenze fisiologiche attese tra i sessi 
> (ad esempio l'emoglobina è normalmente più alta nei maschi). Altre differenze potrebbero invece 
> riflettere bias nella raccolta dati o nella popolazione campionata.

---
## Sezione 8 — Matrice di Correlazione Mirata

Anche se il genere fosse **rimosso** dal modello, il modello potrebbe comunque "vederlo" indirettamente 
attraverso features che **correlano fortemente con il genere** (ad esempio emoglobina, creatinina, BMI).

Questa sezione identifica queste features **proxy** del genere e analizza se le correlazioni tra 
features cliniche e mortalità differiscono tra maschi e femmine.

In [ ]:
# Codifichiamo il genere come variabile numerica: M=0, F=1
df['gender_num'] = (df['gender'] == 'F').astype(int)

# Correlazione di tutte le features numeriche con il genere
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['gender_num', 'patient_id', 'hospital_id']]

corr_with_gender = df[numeric_cols].corrwith(df['gender_num']).dropna().sort_values(key=abs, ascending=False)

print('Top 20 features più correlate con il genere (F=1, M=0):')
print(corr_with_gender.head(20).round(4))

In [ ]:
# Grafico: top 20 features più correlate con il genere
top_corr = corr_with_gender.head(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#DD8452' if v > 0 else '#4C72B0' for v in top_corr.values]
top_corr.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Top 20 Features Più Correlate con il Genere\n(positivo = più alto nelle donne)', fontsize=13)
ax.set_xlabel('Correlazione con genere (F=1, M=0)')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### Perché questo è importante?

Se una feature come l'emoglobina correla fortemente con il genere, un modello può 
**ricostruire il genere** anche senza averlo come input diretto. 
Questo significa che **rimuovere il genere dal modello non elimina il bias**.

In [ ]:
# Correlazione features-mortalità separata per genere
corr_death_m = df_m[numeric_cols].corrwith(df_m['hospital_death']).dropna()
corr_death_f = df_f[numeric_cols].corrwith(df_f['hospital_death']).dropna()

# Uniamo in un dataframe
corr_comparison = pd.DataFrame({
    'Corr_Morte_M': corr_death_m,
    'Corr_Morte_F': corr_death_f
}).dropna()
corr_comparison['Differenza'] = (corr_comparison['Corr_Morte_F'] - corr_comparison['Corr_Morte_M']).round(4)
corr_comparison['Abs_Differenza'] = corr_comparison['Differenza'].abs()
corr_comparison = corr_comparison.sort_values('Abs_Differenza', ascending=False)

print('Features con la maggiore DIFFERENZA di correlazione con la mortalità tra M e F:')
print('(se la correlazione è diversa, il fattore di rischio pesa diversamente per i due sessi)\n')
corr_comparison.head(20).round(4)

In [ ]:
# Scatter plot: correlazione con mortalità M vs F
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(corr_comparison['Corr_Morte_M'], corr_comparison['Corr_Morte_F'],
           alpha=0.4, s=20, color='gray')

# Evidenziamo le features con differenza maggiore
top_diff = corr_comparison.head(10)
ax.scatter(top_diff['Corr_Morte_M'], top_diff['Corr_Morte_F'],
           alpha=0.9, s=50, color='red', zorder=5)

for feat in top_diff.index:
    ax.annotate(feat, (top_diff.loc[feat, 'Corr_Morte_M'], top_diff.loc[feat, 'Corr_Morte_F']),
                fontsize=7, alpha=0.8)

# Linea diagonale (correlazione uguale per M e F)
lim = max(abs(corr_comparison['Corr_Morte_M'].max()), abs(corr_comparison['Corr_Morte_F'].max())) + 0.05
ax.plot([-lim, lim], [-lim, lim], 'k--', alpha=0.5, label='Uguale per M e F')

ax.set_xlabel('Correlazione con mortalità — Maschi', fontsize=11)
ax.set_ylabel('Correlazione con mortalità — Femmine', fontsize=11)
ax.set_title('Confronto: Correlazione Feature-Mortalità tra Maschi e Femmine\n(punti rossi = maggiori differenze)', fontsize=12)
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

> **Interpretazione**: i punti lontani dalla diagonale rappresentano features che hanno un 
> **impatto diverso sulla mortalità** per maschi e femmine. Se il modello apprende una relazione "media", 
> farà errori sistematici per uno dei due gruppi. Questa è una **fonte diretta di bias algoritmico**.

---
## Sezione 9 — PCA per Genere

La **PCA (Analisi delle Componenti Principali)** è una tecnica che "comprime" molte features 
in poche dimensioni, mantenendo la maggior parte dell'informazione.

Se il genere è visibile nella PCA (cioè se maschi e femmine formano gruppi separati), 
significa che l'informazione di genere è **pervasiva** nei dati clinici.

In [ ]:
# Carichiamo i dati di training già standardizzati
train_labels = pd.read_csv(path_to_data_folder + 'train_labels.csv')

print(f'Train features: {train_features.shape}')
print(f'Train labels: {train_labels.shape}')

In [ ]:
# Eseguiamo la PCA sulle features di training
# Rimuoviamo eventuali NaN rimasti
train_clean = train_features.fillna(0)

pca = PCA(n_components=10, random_state=42)
pca_result = pca.fit_transform(train_clean)

print(f'Varianza spiegata dalle prime 10 componenti: {pca.explained_variance_ratio_.sum()*100:.1f}%')
print(f'Varianza spiegata per componente:')
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {var*100:.1f}%')

In [ ]:
# Identifichiamo il genere nel training set (dalle colonne one-hot)
if 'gender_F' in train_features.columns:
    train_gender = train_features['gender_F'].map({1: 'F', 0: 'M'})
else:
    train_gender = pd.Series(['Sconosciuto'] * len(train_features))

# Identifichiamo la mortalità
if 'hospital_death' in train_labels.columns:
    train_death = train_labels['hospital_death'].map({0: 'Sopravvissuto', 1: 'Deceduto'})
else:
    train_death = train_labels.iloc[:, 0].map({0: 'Sopravvissuto', 1: 'Deceduto'})

In [ ]:
# Scatter plot PC1 vs PC2 — colorato per genere
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Per genere
for g, color in GENDER_PALETTE.items():
    mask = train_gender == g
    axes[0].scatter(pca_result[mask, 0], pca_result[mask, 1],
                    alpha=0.1, s=5, color=color, label=g)

axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('PCA — Colorato per Genere', fontsize=13)
axes[0].legend(markerscale=5, fontsize=11)

# Per mortalità
death_palette = {'Sopravvissuto': '#2ecc71', 'Deceduto': '#e74c3c'}
for label, color in death_palette.items():
    mask = train_death == label
    axes[1].scatter(pca_result[mask, 0], pca_result[mask, 1],
                    alpha=0.1, s=5, color=color, label=label)

axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title('PCA — Colorato per Mortalità', fontsize=13)
axes[1].legend(markerscale=5, fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Varianza spiegata cumulativa
fig, ax = plt.subplots(figsize=(8, 4))

cumvar = np.cumsum(pca.explained_variance_ratio_) * 100
ax.bar(range(1, 11), pca.explained_variance_ratio_ * 100, alpha=0.6, label='Singola componente')
ax.plot(range(1, 11), cumvar, 'ro-', label='Cumulativa')

ax.set_xlabel('Componente Principale')
ax.set_ylabel('Varianza Spiegata (%)')
ax.set_title('Varianza Spiegata dalla PCA', fontsize=13)
ax.set_xticks(range(1, 11))
ax.legend()
plt.tight_layout()
plt.show()

> **Commento sulla PCA**: osservate se nel grafico PC1 vs PC2 i punti blu (M) e arancioni (F) 
> si sovrappongono completamente o se formano cluster parzialmente separati. 
> Se c'è separazione visibile, il modello potrà facilmente "ricostruire" il genere dalle features, 
> anche senza accedervi direttamente.

---
## Sezione 10 — Audit di Rappresentatività

Il dataset è rappresentativo della popolazione reale di terapia intensiva? 
Confrontiamo le proporzioni del dataset con valori di riferimento approssimati 
della popolazione ICU negli Stati Uniti.

In [ ]:
# Tabella incrociata: genere x età x etnia
cross_tab = df.groupby(['gender', 'age_group', 'ethnicity']).size().reset_index(name='count')
cross_tab['pct'] = (cross_tab['count'] / len(df) * 100).round(2)

# Mostriamo i gruppi più rappresentati
print('=== Top 20 gruppi più rappresentati ===')
print(cross_tab.sort_values('count', ascending=False).head(20).to_string(index=False))

In [ ]:
# Gruppi meno rappresentati (potenziali problemi di bias)
print('=== Top 20 gruppi MENO rappresentati ===')
print(cross_tab.sort_values('count', ascending=True).head(20).to_string(index=False))

In [ ]:
# Confronto con valori di riferimento approssimati della popolazione ICU USA
# Fonti: letteratura epidemiologica ICU (valori approssimati)

ref_data = {
    'Dimensione': ['Genere M', 'Genere F',
                   'Caucasian', 'African American', 'Hispanic', 'Asian', 'Native American', 'Other',
                   'Età media (anni)'],
    'Dataset GOSSIS (%)': [
        round(100 * (df['gender'] == 'M').mean(), 1),
        round(100 * (df['gender'] == 'F').mean(), 1),
        round(100 * (df['ethnicity'] == 'Caucasian').mean(), 1),
        round(100 * (df['ethnicity'] == 'African American').mean(), 1),
        round(100 * (df['ethnicity'] == 'Hispanic').mean(), 1),
        round(100 * (df['ethnicity'] == 'Asian').mean(), 1),
        round(100 * (df['ethnicity'] == 'Native American').mean(), 1),
        round(100 * (df['ethnicity'] == 'Other/Unknown').mean(), 1) if 'Other/Unknown' in df['ethnicity'].values else round(100 * (df['ethnicity'].isin(['Other/Unknown', 'Other'])).mean(), 1),
        round(df['age'].mean(), 1)
    ],
    'Rif. popolazione ICU USA (%)': [
        55.0,  # Maschi leggermente sovra-rappresentati nelle ICU
        45.0,
        65.0,  # Caucasici: circa 65% popolazione ICU
        15.0,  # African American: circa 15%
        10.0,  # Hispanic: circa 10%
        4.0,   # Asian: circa 4%
        1.5,   # Native American: circa 1.5%
        4.5,   # Other
        62.0   # Età media ICU
    ]
}

ref_df = pd.DataFrame(ref_data)
ref_df['Differenza'] = ref_df['Dataset GOSSIS (%)'] - ref_df['Rif. popolazione ICU USA (%)']
ref_df['Differenza'] = ref_df['Differenza'].round(1)

print('=== Confronto Dataset vs Popolazione ICU di Riferimento ===')
print(ref_df.to_string(index=False))

In [ ]:
# Visualizzazione delle sotto-rappresentazioni
fig, ax = plt.subplots(figsize=(10, 5))

ref_plot = ref_df.iloc[:-1].copy()  # escludiamo l'età media
x = np.arange(len(ref_plot))
width = 0.35

ax.bar(x - width/2, ref_plot['Dataset GOSSIS (%)'], width, label='Dataset GOSSIS', color='#4C72B0')
ax.bar(x + width/2, ref_plot['Rif. popolazione ICU USA (%)'], width, label='Rif. ICU USA', color='#DD8452', alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels(ref_plot['Dimensione'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Proporzione (%)')
ax.set_title('Rappresentatività del Dataset vs Popolazione ICU USA', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: conteggi per genere x età x etnia (solo le etnie principali)
main_ethnicities = ['Caucasian', 'African American', 'Hispanic', 'Asian', 'Native American']
df_main_eth = df[df['ethnicity'].isin(main_ethnicities)].copy()

pivot_counts = df_main_eth.groupby(['age_group', 'ethnicity', 'gender']).size().reset_index(name='n')

# Creiamo una colonna combinata genere+età per una heatmap leggibile
pivot_table = pivot_counts.pivot_table(index='ethnicity', columns=['gender', 'age_group'],
                                        values='n', fill_value=0)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot_table.fillna(0).astype(int), annot=True, fmt='d', cmap='YlOrBr', ax=ax)
ax.set_title('Conteggi per Etnia, Genere e Fascia di Età', fontsize=13)
plt.tight_layout()
plt.show()

# Evidenziamo le intersezioni sotto-rappresentate
print('\n=== Intersezioni con meno di 100 pazienti (potenziale problema di rappresentatività) ===')
small_groups = cross_tab[cross_tab['count'] < 100].sort_values('count')
print(f'Numero di intersezioni sotto-rappresentate: {len(small_groups)}')
print(small_groups.head(20).to_string(index=False))

> **Osservazione chiave**: alcune intersezioni (ad esempio Donna + African American + >80 anni, 
> o Donna + Native American + qualsiasi età) hanno pochissimi pazienti. 
> Un modello addestrato su questi dati avrà prestazioni particolarmente inaffidabili per questi sottogruppi.
> 
> Inoltre, il dataset **sovra-rappresenta** i pazienti caucasici rispetto alla distribuzione reale 
> della popolazione ICU statunitense, e **sotto-rappresenta** le minoranze etniche.

---
## Sezione 11 — Riepilogo e Messaggi Chiave

Ecco un riassunto di tutto ciò che abbiamo scoperto esplorando il dataset GOSSIS.

---

### 1. Il dataset sembra bilanciato per sesso, ma non lo è nelle intersezioni

La distribuzione complessiva (54% M, 46% F) appare ragionevole. Tuttavia, quando si incrociano 
genere, età ed etnia, emergono numerosi sottogruppi con pochissimi pazienti. 
Un modello non può imparare bene da gruppi con meno di 100 osservazioni.

### 2. Le donne hanno una mortalità leggermente più alta

Il tasso di mortalità femminile (~8.8%) è leggermente superiore a quello maschile (~8.4%). 
Questa differenza, apparentemente piccola, potrebbe amplificarsi nelle previsioni del modello, 
soprattutto se le features predittive si comportano diversamente per i due sessi.

### 3. Enormi differenze di valori mancanti potrebbero riflettere bias nell'accesso ai test

Molte features cliniche (in particolare quelle legate ai gas ematici) hanno il 60-70% di valori mancanti. 
Se la percentuale di mancanza differisce tra M e F, questo suggerisce che le donne potrebbero 
ricevere meno esami diagnostici in terapia intensiva — un **bias implicito nell'accesso alle cure** 
che si trasferisce nei dati e poi nel modello.

### 4. Molte features cliniche correlano con il genere

Features come emoglobina, creatinina e BMI hanno una forte correlazione con il sesso. 
Questo significa che anche **rimuovendo la variabile genere** dal modello, 
il modello può ricostruirla indirettamente attraverso queste features proxy. 
La semplice rimozione del genere **non è sufficiente** per eliminare il bias.

### 5. La PCA e la struttura dei dati

L'analisi PCA rivela quanto l'informazione di genere sia "distribuita" nelle features cliniche. 
Se i cluster per genere sono visibili anche nelle prime componenti principali, 
conferma che il genere è una caratteristica pervasiva dei dati.

---

### Cosa faremo nelle prossime fasi del workshop

Nelle prossime sessioni:
- **Fase 1-2**: Addestreremo modelli predittivi (Random Forest, XGBoost) e valuteremo le loro prestazioni
- **Fase 3**: Confronteremo le metriche di equità (fairness) tra maschi e femmine
- **Fase 4**: Useremo tecniche di spiegabilità (SHAP) per capire *come* il modello usa le features legate al genere

> *L'obiettivo finale non è eliminare il bias, ma **renderlo visibile**, misurabile e gestibile 
> nella pratica clinica quotidiana.*